In [ ]:
import sys
import os
sys.path.append(os.path.abspath(".."))  # This brings 'src' into the path

import yaml
config = yaml.safe_load(open('../config.yaml', 'r'))

import os
os.environ["CUDA_VISIBLE_DEVICES"] = '6'
import torch
import torch.nn.functional as F

from src.model.models_dsfno_3d import DSFNO
from src.dataloader.dataloader_3d import lazy_dataset_sr as dataset_sr
import numpy as np
from matplotlib.colors import LogNorm
import matplotlib.pyplot as plt
import random
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
dsfno = DSFNO(in_channel=5, 
                    modes=config['dsfno']['modes'],
                    n_channels=config['dsfno']['n_channels'],
                    n_residual_blocks=config['dsfno']['n_residual_blocks'],
                    n_operator_blocks=config['dsfno']['n_operator_blocks'], 
                    apply_constraint=config['dsfno']['apply_constraint']).to(device)

In [ ]:
max_samples = 30
dataset = dataset_sr(max_samples=max_samples)

In [ ]:
dsfno.load_state_dict(torch.load('../model/dsfno_model', weights_only=True))

In [ ]:

print(f"\n=== 3D DSFNO (modes={config['dsfno']['modes']},n_channels={config['dsfno']['n_channels']}) ===")
print(f"\n=== n_residual_blocks={config['dsfno']['n_residual_blocks']}, n_operator_blocks={config['dsfno']['n_operator_blocks']}, apply_constraint={config['dsfno']['apply_constraint']}===")

i = random.sample(range(max_samples), 1)[0]
hr_state, lr_state = dataset[5]

# Prepare input
x = torch.unsqueeze(lr_state, 0).to(device)
upsample_factor = 4

with torch.no_grad():

    out = dsfno.conv1(x)
    for i, layer in enumerate(dsfno.res_blocks):
        out = layer(out)
    out = dsfno.conv2(out)
    after_res = out

    out = torch.nn.functional.interpolate(out, scale_factor=upsample_factor, mode='trilinear', align_corners=False)
    after_interpolate = out

    for i, layer in enumerate(dsfno.fno_blocks):
        out = layer(out)
    after_fno = out

    out = out.permute(0, 2, 3, 4, 1)
    out = dsfno.fc1(out)
    out = F.gelu(out)
    out = dsfno.fc2(out)
    out = out.permute(0, 4, 1, 2, 3)
    after_fc = out

    if dsfno.apply_constraint:
        out = dsfno.constraint(x, out, upsample_factor)
    after_constraint = out

del out

In [ ]:
figs = [torch.squeeze(fig,0) for fig in [x, after_res, after_interpolate, after_fno, after_fc, after_constraint]]
names = ['input', 'after_res', 'after_interpolate', 'after_fno', 'after_fc', 'after_constraint']
for i, state in enumerate(zip(figs,names)):
    state_image = state[0]
    name = state[1]
    print(name, state_image.shape)

In [ ]:
scale_factor = 4
sr_factor = 4
fig, axes = plt.subplots(6, 3, figsize=(12,24))

# plot cuts at this z level
z_level = 60
z_level_reduced = z_level // scale_factor
z_level_sr = z_level_reduced * sr_factor

figs = [torch.squeeze(fig,0).cpu().numpy() for fig in [x, after_res, after_interpolate, after_fno, after_fc, after_constraint]]
indexes = [z_level_reduced, z_level_reduced, z_level_sr, z_level_sr, z_level_sr, z_level_sr]
names = ['input', 'after_res', 'after_interpolate', 'after_fno', 'after_fc', 'after_constraint']

for ax in axes.flat:
    ax.tick_params(left=False, bottom=False, labelleft=False, labelbottom=False)
    for spine in ax.spines.values():
        spine.set_visible(False)

figures = []
for i, state in enumerate(zip(figs,names, indexes)):
    state_image = state[0]
    name = state[1]
    z_level = state[2]

    figures.append(axes[i,0].imshow(state_image[0, :, :, z_level].T, origin = "lower", extent = [0, 1, 0, 1], norm = LogNorm()))
    axes[i,0].set_ylabel(f"{name}")
    figures.append(axes[i,1].imshow(np.sqrt(state_image[1, :, :, z_level]**2 + state_image[2, :, :, z_level]**2 + state_image[3, :, :, z_level]**2).T, origin = "lower", extent = [0, 1, 0, 1]))
    figures.append(axes[i,2].imshow(state_image[4, :, :, z_level].T, origin = "lower", extent = [0, 1, 0, 1], norm = LogNorm()))



axes[0,0].set_title("Density")
axes[0,1].set_title("Velocity")
axes[0,2].set_title("Pressure")

In [ ]:
i  = 5
scale_factor = 4
sr_factor = 4
fig, axes = plt.subplots(3, 3, figsize=(12,12))

# plot cuts at this z level
z_level = 60
z_level_reduced = z_level // scale_factor
z_level_sr = z_level_reduced * sr_factor

for ax in axes.flat:
    ax.tick_params(left=False, bottom=False, labelleft=False, labelbottom=False)
    for spine in ax.spines.values():
        spine.set_visible(False)
        
hr_state, lr_state_tensor = dataset[i]

with torch.no_grad():
    sr_state = torch.squeeze(dsfno(torch.unsqueeze(lr_state_tensor, 0).to(device), sr_factor),0).cpu().detach().numpy()
hr_state, lr_state = hr_state.numpy(), lr_state_tensor.numpy()
figures = []

figures.append(axes[0,0].imshow(lr_state[0, :, :, z_level_reduced].T, origin = "lower", extent = [0, 1, 0, 1], norm = LogNorm()))
axes[0,0].set_ylabel(f"{i}")
figures.append(axes[0,1].imshow(np.sqrt(lr_state[1, :, :, z_level_reduced]**2 + lr_state[2, :, :, z_level_reduced]**2 + lr_state[2, :, :, z_level_reduced]**2).T, origin = "lower", extent = [0, 1, 0, 1]))
figures.append(axes[0,2].imshow(lr_state[4, :, :, z_level_reduced].T, origin = "lower", extent = [0, 1, 0, 1], norm = LogNorm()))

figures.append(axes[1,0].imshow(hr_state[0, :, :, z_level].T, origin = "lower", extent = [0, 1, 0, 1], norm = LogNorm()))
figures.append(axes[1,1].imshow(np.sqrt(hr_state[1, :, :, z_level]**2 + hr_state[2, :, :, z_level]**2 + hr_state[3, :, :, z_level]**2).T, origin = "lower", extent = [0, 1, 0, 1]))
figures.append(axes[1,2].imshow(hr_state[4, :, :, z_level].T, origin = "lower", extent = [0, 1, 0, 1], norm = LogNorm()))

figures.append(axes[2,0].imshow(sr_state[0, :, :, z_level_sr].T, origin = "lower", extent = [0, 1, 0, 1], norm = LogNorm()))
figures.append(axes[2,1].imshow(np.sqrt(sr_state[1, :, :, z_level_sr]**2 + sr_state[2, :, :, z_level_sr]**2 + sr_state[3, :, :, z_level_sr]**2 ).T, origin = "lower", extent = [0, 1, 0, 1]))
figures.append(axes[2,2].imshow(sr_state[4, :, :, z_level_sr].T, origin = "lower", extent = [0, 1, 0, 1] , norm = LogNorm()))

for figs in figures:
    fig.colorbar(figs)
axes[0,0].set_title("Density")
axes[0,1].set_title("Velocity")
axes[0,2].set_title("Pressure")